In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso


import classification_rf

from importlib import reload
reload(classification_rf)

from classification_rf import RandomForestClassifier621

np.random.seed(24)

import rf_wrapper_functions as rfwrap
from collections import namedtuple
Dataset = namedtuple('Dataset', ['X_train', 'X_test', 'y_train', 'y_test'])
Params = namedtuple('Params', ['min_samples_leaf', 'max_features', 'max_depth', 'force_label_split', 'oob_score'])


In [7]:
from sklearn.impute import SimpleImputer

#import Cleveland Data
processed_cleveland_data = pd.read_csv("Processed Data/processed.cleveland.data", delimiter=",", na_values=["?"], header = None).to_numpy()
processed_cleveland_data = np.delete(processed_cleveland_data, [11,12], axis=1)

#Fill in missing values
imputer = SimpleImputer(strategy="median")
cleveland = imputer.fit_transform(processed_cleveland_data)

#add label column
cleveland_avg = np.average(cleveland[:,-1])
cleveland = np.insert(cleveland, -1, [cleveland_avg]*len(cleveland), axis = 1)

#---------------------------

processed_hungarian_data = pd.read_csv("Processed Data/processed.hungarian.data", delimiter=",", na_values=["?"], header = None).to_numpy()
processed_hungarian_data = np.delete(processed_hungarian_data, [11,12], axis=1)
imputer = SimpleImputer(strategy="median")
hungary = imputer.fit_transform(processed_hungarian_data)

#add label column
hungary_avg = np.average(hungary[:,-1])
hungary = np.insert(hungary, -1, [hungary_avg]*len(hungary), axis = 1)

#---------------------------

processed_switzerland_data = pd.read_csv("Processed Data/processed.switzerland.data", delimiter=",", na_values=["?"], header = None).to_numpy()
processed_switzerland_data = np.delete(processed_switzerland_data, [11,12], axis=1)
imputer = SimpleImputer(strategy="median")
switzerland = imputer.fit_transform(processed_switzerland_data)

#add label column
switzerland_avg = np.average(switzerland[:,-1])
switzerland = np.insert(switzerland, -1, [switzerland_avg]*len(switzerland), axis = 1)

#---------------------------

processed_va_data = pd.read_csv("Processed Data/processed.va.data", delimiter=",", na_values=["?"], header = None).to_numpy()
processed_va_data = np.delete(processed_va_data, [11,12], axis=1)
imputer = SimpleImputer(strategy="median")
va = imputer.fit_transform(processed_va_data)

#add label column
va_avg = np.average(va[:,-1])
va = np.insert(va, -1, [va_avg]*len(va), axis = 1)

#data = np.vstack((processed_cleveland_data, processed_hungarian_data, processed_switzerland_data, processed_va_data))
data = np.vstack((cleveland, hungary, switzerland, va))

labels = [cleveland_avg, hungary_avg, switzerland_avg, va_avg]

In [8]:
X = data[:,0:-1]
y = data[:,-1]
y[y != 0] = 1
y = y.astype(int)

In [9]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(sampling_strategy='auto', random_state=24)

for label in labels:
    indices = np.where(X[:, -1] == label)[0]

    X_resampled, y_resampled = ros.fit_resample(X[indices], y[indices])

    X = np.delete(X, indices, axis=0)
    y = np.delete(y, indices, axis=0)

    X = np.vstack((X, X_resampled))      # ← assign back
    y = np.concatenate((y, y_resampled)) # ← concatenate, not vstack

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

In [ ]:
dataset = Dataset(X_train, X_test, y_train, y_test)
parameters = Params(3, "sqrt", 25, False, False)

accuracy_score, rf, source_score, source_size = rfwrap.forest_wrapper(n_estimators=100, depth=20, dataset=dataset, parameters=parameters, plot=True)


In [ ]:
#Source Accuracy
depth = np.arange(15)

for i in range(len(source_score[0])):
    plt.plot(depth, source_score[:,i], label = f"Source {i} (n =  {source_size[i]})")

plt.plot(depth, accuracy_score, color = 'black', linestyle = "dashed", label = "total accuracy")
plt.xlabel('d_0 depth')
plt.ylabel('R^2 accuracy')
plt.legend()
#plt.ylim(0,1)
plt.title("Sourced accuracy on Lasso trimmed Data by D0 depth")

NameError: name 'source_score' is not defined

In [ ]:
X_va = X_res[X[:,-1] == 0]
y_va = y_res[:,-1]
y_va[y_va != 0] = 1
y_va = y_va.astype(int)

print(np.shape(X_va))

X_va_train, X_va_test, y_va_train, y_va_test = train_test_split(X_va, y_va, test_size=0.2, random_state=0, stratify=X_va[:,-1])
source_data = Dataset(X_va_train, X_va_test, y_va_train, y_va_test)
accuracy_score, rf, source_score, source_size = rfwrap.forest_wrapper(n_estimators=100, depth=10, dataset=source_data, parameters=parameters)

predictions = rf[0].predict(X_va_test)

print(predictions, "\n", y_va_test)

print(rf[0].score(X_va_test, y_va_test))

(200, 12)
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.] 
 [1 1 1 1 0 0 0 1 0 1 0 0 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 1 1 0 1 1 1 1 1
 1 1 1]
0.75
